# Running the Models with Various Preprocessing Techniques and K-Fold Cross-Validation

## Setup

In [1]:
import os

# Get the directory of the current notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

print(f"Project root directory: {PROJECT_ROOT}")

Project root directory: /mnt/hdd/Faculdade/2025.1/Processamento de Imagens/breast_cancer_analysis


In [2]:
# for importing utils from python scripts in parent directories

import sys
sys.path.append('../utils')
sys.path.append('..')

In [3]:
# suppress TensorFlow warnings and logs

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0 = all, 1 = info, 2 = warning, 3 = error

import warnings
warnings.filterwarnings('ignore')

# If you use logging, also suppress it:
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

In [4]:
import tensorflow as tf


gpus = tf.config.experimental.list_physical_devices('GPU')
print("GPUs available:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Defining the image processing function

In [5]:
import numpy as np
import cv2

def preprocess_images(df, preproc_fn):
    X = []
    y = []
    for idx, row in df.iterrows():
        img_path = row['image_path']
        label = row['label']
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Could not read {img_path}")
            continue
        processed = preproc_fn(img)
        X.append(processed)
        y.append(label)
    X = np.stack(X)
    y = np.stack(y)
    return X, y

## Get the training and testing data

In [6]:
# if not using folds during training, load the train and test sets directly

import pandas as pd

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_split.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_split.csv'))

print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

Train set: 205 samples
Test set: 69 samples


In [7]:
label_mapping = {
    '1': 0,
    '2': 1,
    '3': 2,
    '4a': 3,
    '4b': 4,
    '4c': 5,
    '5': 6,
    '6': 7
}

train_df['label'] = train_df['label'].map(label_mapping)
test_df['label'] = test_df['label'].map(label_mapping)

print('Label mapping applied. Unique train labels:', train_df['label'].unique())
print('Label mapping applied. Unique test labels:', test_df['label'].unique())
print(train_df['label'].isnull().sum(), test_df['label'].isnull().sum())
print(train_df['label'].unique(), test_df['label'].unique())

Label mapping applied. Unique train labels: [0 2 1 3 6 5 4 7]
Label mapping applied. Unique test labels: [2 5 1 0 7 6 3 4]
0 0
[0 2 1 3 6 5 4 7] [2 5 1 0 7 6 3 4]


## Running the Models

### Model Running Function

In [8]:
import numpy as np
from keras.utils import to_categorical
from keras import backend as K

def run_model_with_preprocessing(
    train_df, test_df, preproc_fn, model_name, model_fn, num_classes=8, batch_size=8, epochs=15
):
    X_train, y_train = preprocess_images(train_df, preproc_fn)
    X_test, y_test = preprocess_images(test_df, preproc_fn)
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0

    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)

    if model_name.lower() in ['custom cnn']:
        input_shape = X_train.shape[1:]
        X_tr, X_te = X_train, X_test
    else:
        X_train_3ch = np.repeat(X_train, 3, axis=-1)
        X_test_3ch = np.repeat(X_test, 3, axis=-1)
        input_shape = X_train_3ch.shape[1:]
        X_tr, X_te = X_train_3ch, X_test_3ch
        batch_size = 4

    model = model_fn(input_shape=input_shape, num_classes=num_classes, loss='categorical_crossentropy')
    history = model.fit(
        X_tr, y_train,
        validation_data=(X_te, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )
    val_accuracies = history.history['val_accuracy']
    best_epoch = int(np.argmax(val_accuracies)) + 1
    best_val_acc = float(np.max(val_accuracies))

    return best_val_acc, best_epoch, history.history, model, X_te, test_df

### Auxiliary Functions

In [9]:
def merge_train_test(train_df, test_df):
    train_df['set'] = 'train'
    test_df['set'] = 'test'
    merged_df = pd.concat([train_df, test_df], ignore_index=True)
    return merged_df

In [10]:
import os

def check_if_model_exists(preproc_id, model_name, combo_str):
    results_pred_dir = os.path.join(DATA_DIR, 'results', 'history', preproc_id, model_name)
    history_file = os.path.join(results_pred_dir, f"history_{combo_str.replace(', ', '_').replace('=', '-')}.csv")
    history_exists = os.path.exists(history_file)
    
    predictions_dir = os.path.join(DATA_DIR, 'results', 'predictions', preproc_id, model_name)
    prediction_file = os.path.join(predictions_dir, f"{combo_str.replace(', ', '_').replace('=', '-')}.csv")    
    prediction_exists = os.path.exists(prediction_file)
    
    return history_exists and prediction_exists

### Running the models

In [11]:
import itertools
from utils.preprocessing import preprocessing_methods
from utils.models import MODEL_BUILDERS
import time
from sklearn.model_selection import StratifiedKFold
import pandas as pd

fold = 4 # set to the amount of folds used (0 for no folds)
batch_size = 8
run_skip = True # if True, will skip already ran models

for preproc_id, preproc_info in preprocessing_methods.items():
    param_names = list(preproc_info['params'].keys())
    param_values = [preproc_info['params'][k] for k in param_names]
    for param_combo in itertools.product(*param_values):
        param_dict = dict(zip(param_names, param_combo))
        def preproc_fn(img, func=preproc_info['func'], params=param_dict):
            return func(img, **params)
        combo_str = ', '.join([f"{k}={v}" for k, v in param_dict.items()])
        print(f"\n=== Preprocessing: {preproc_id} ({combo_str}) ===")
        for model_name, model_fn in MODEL_BUILDERS.items():
            try:
                print(f"--> Current Model: {model_name} <--")
                if run_skip and check_if_model_exists(preproc_id, model_name, combo_str):
                    print(f"Skipping {preproc_id} [{model_name} - {combo_str}] as it has already been run.")
                    continue
                model_time = time.time()

                if fold == 0:
                    # not merging train and test sets
                    train_set, test_set = train_df, test_df

                    best_val_acc, best_epoch, history_dict, model, X_te, test_set = run_model_with_preprocessing(
                        train_set, test_set, preproc_fn, model_name, model_fn, num_classes=8, batch_size=batch_size, epochs=15
                    )
                    print(f"Best val accuracy for {preproc_id} [{model_name} - {combo_str}]: {best_val_acc:.4f} at epoch {best_epoch}")

                else:
                    # if using folds, only get the best fold data
                    merged_df = merge_train_test(train_df, test_df)
                    X = merged_df['image_path']
                    y = merged_df['label']
                    skf = StratifiedKFold(n_splits=fold, shuffle=True, random_state=42)
                    fold_results = []
                    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                        print(f"Running fold {fold_idx+1}/{fold}...")
                        train_set = merged_df.iloc[train_idx].reset_index(drop=True)
                        test_set = merged_df.iloc[test_idx].reset_index(drop=True)
                        best_val_acc, best_epoch, history_dict, model, X_te, test_set = run_model_with_preprocessing(
                            train_set, test_set, preproc_fn, model_name, model_fn, num_classes=8, batch_size=batch_size, epochs=15
                        )
                        fold_results.append({
                            'fold': fold_idx+1,
                            'best_val_acc': best_val_acc,
                            'best_epoch': best_epoch
                        })
                        print(f"Fold {fold_idx+1}: Best val accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

                    # get the best fold (highest val accuracy)
                    best_fold = max(fold_results, key=lambda x: x['best_val_acc'])
                    print(f"Best fold for {preproc_id} [{model_name} - {combo_str}]: Fold {best_fold['fold']} with val accuracy {best_fold['best_val_acc']:.4f} at epoch {best_fold['best_epoch']}")
                    
                    # save the model's predictions and labels for further analysis
                    results_pred_dir = os.path.join(DATA_DIR, 'results', 'predictions', preproc_id, model_name)
                    os.makedirs(results_pred_dir, exist_ok=True)
                    csv_path = os.path.join(results_pred_dir, f"{combo_str.replace(', ', '_').replace('=', '-')}.csv")
                    y_pred_prob = model.predict(X_te, batch_size=batch_size)
                    y_pred = np.argmax(y_pred_prob, axis=1)
                    df = pd.DataFrame({
                        'y_true': test_set['label'].values,
                        'y_pred': y_pred
                    })
                    df['y_pred_probability'] = y_pred_prob.tolist()
                    for i in range(y_pred_prob.shape[1]):
                        df[f'prob_class_{i}'] = y_pred_prob[:, i]
                    df.to_csv(csv_path, index=False)

                    # save best fold results to CSV for run_skip
                    results_dir = os.path.join(DATA_DIR, 'results', 'history', preproc_id, model_name)
                    os.makedirs(results_dir, exist_ok=True)
                    history_file = os.path.join(results_dir, f"history_{combo_str.replace(', ', '_').replace('=', '-')}.csv")
                    pd.DataFrame([best_fold]).to_csv(history_file, index=False)
                    
                    K.clear_session()
                    del model
                    import gc; gc.collect()
            except tf.errors.ResourceExhaustedError as e:
                print(f"GPU memory error detected for {preproc_id} [{model_name} - {combo_str}]. Exiting for external restart...")
                os._exit(1) # exit the process immediately to allow external restart
                
            except Exception as e:
                print(f"Error running {preproc_id} [{model_name} - {combo_str}]: {e}")
                continue

            end_model_time = time.time()
            elapsed = end_model_time - model_time
            h = int(elapsed // 3600)
            m = int((elapsed % 3600) // 60)
            s = int(elapsed % 60)
            parts = []
            if h > 0:
                parts.append(f"{h}h")
            if m > 0:
                parts.append(f"{m}min")
            if s > 0 or not parts:
                parts.append(f"{s}sec")
            print(f"Time taken for {preproc_id} [{model_name}]: {' '.join(parts)}\n")


=== Preprocessing: denoise (kernel_size=(3, 3), sigma=0) ===
--> Current Model: custom cnn <--
Skipping denoise [custom cnn - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: resnet <--
Skipping denoise [resnet - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: densenet <--
Skipping denoise [densenet - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: efficientnet <--
Skipping denoise [efficientnet - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: mobilenetv3 <--
Skipping denoise [mobilenetv3 - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: inception <--
Skipping denoise [inception - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: nasnet <--
Skipping denoise [nasnet - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: bcnet <--
Skipping denoise [bcnet - kernel_size=(3, 3), sigma=0] as it has alread

Fold 1: Best val accuracy: 0.1304 at epoch 1
Running fold 2/4...


Fold 2: Best val accuracy: 0.1884 at epoch 12
Running fold 3/4...


Fold 3: Best val accuracy: 0.1471 at epoch 12
Running fold 4/4...


Fold 4: Best val accuracy: 0.1912 at epoch 11
Best fold for denoise [bcnet - kernel_size=(3, 3), sigma=3]: Fold 4 with val accuracy 0.1912 at epoch 11


...

9/9 [==============================] - 2s 35ms/step


Time taken for denoise [densenet]: 2min 12sec

--> Current Model: efficientnet <--
Running fold 1/4...


Fold 1: Best val accuracy: 0.1449 at epoch 1
Running fold 2/4...


Fold 2: Best val accuracy: 0.1304 at epoch 1
Running fold 3/4...


Fold 3: Best val accuracy: 0.1471 at epoch 2
Running fold 4/4...
